In [5]:
import pandas as pd 
import numpy as np
import wrds

In [ ]:
db = wrds.Connection(username = 'hcdb')

In [ ]:
# Get Compustat Annual Information

funda = db.raw_sql('''
                   SELECT c.cik, f.gvkey, f.cusip, f.datadate, f.fyear, f.sic, f.naics,
                          f.sale,f.revt,f.cogs,f.xsga,f.dp,f.xrd,f.xad,f.ib,f.ebitda,f.ebit,f.nopi,f.spi,f.pi,f.txp,f.ni,f.txfed,f.txfo,f.txt,f.xint,
                          f.capx,f.oancf,f.dvt,f.ob,f.gdwlia,f.gdwlip,f.gwo,
                          f.rect,f.act,f.che,f.ppegt,f.invt,f.at,f.aco,f.intan,f.ao,f.ppent,f.gdwl,f.fatb,f.fatl,
                          f.lct,f.dlc,f.dltt,f.lt,f.dm,f.dcvt,f.cshrc,f.dcpstk,f.pstk,f.ap,f.lco,f.lo,f.drc,f.drlt,f.txdi,
                          f.ceq,f.scstkc,f.emp,f.csho,
                          f.prcc_f
                   FROM comp.company AS c 
                   JOIN comp.funda AS f
                   ON f.gvkey = c.gvkey
                   WHERE datadate >='1975-01-01'
                   AND f.indfmt='INDL'
                   AND f.datafmt='STD'
                   AND f.popsrc='D'
                   AND f.consol='C'
                   ''').reset_index(drop=True)

# this table actually have two tables comp.company and comp.funda, and  gvkey, sic and
# naics are in both so have to state from which table
#  drop=True discards the old index instead of inserting it back as a new column named index.             
                          

In [ ]:
# Clean Compustat Annual Information

# In-SQL code 

funda['sic2'] = funda['sic'].str.slice(0,2)
funda['cnum'] = funda['cusip'].str.strip().str.slice(0,6) # remove first the whitespaces if any then keep 0 to 6
funda['prcc_f'] = funda['prcc_f'].abs()
funda['mve_f'] = funda['prcc_f']*funda['csho']
funda = funda.dropna(subset=['at','prcc_f','ni'])
funda['datadate'] = pd.to_datetime(funda['datadate'])

# sort and clean up

funda = funda.sort_values(by=['gvkey','datadate']) # by sorting we enable it to keep the first one
funda = funda.drop_duplicates(subset = ['gvkey','datadate'])

#  prep for clean-up and using time series of variables

funda['count'] = funda.groupby('gvkey').cumcount() + 1

# do some clean up, several of these variables have lots of missing values

funda['dr'] = np.where(funda['drc'].notna() & funda['drlt'].notna(), funda['drc'] + funda['drlt'], # since x + NA = NA in python
                       np.where(funda['drc'].notna() & funda['drlt'].isnull(), funda['drc'],
                                np.where(funda['drlt'].notna() & funda['drc'].isnull(), funda['drlt'], np.nan)))  

funda['dc'] = np.where(funda['dcvt'].isnull() & funda['dcpstk'].notna() & funda['pstk'].notna() & (funda['dcpstk'] > funda['pstk']), funda['dcpstk'] - funda['pstk'],
               np.where(funda['dcvt'].isnull() & funda['dcpstk'].notna() & funda['pstk'].isnull(), funda['dcpstk'],
                        funda['dcvt']))

funda['xint0'] = funda['xint'].fillna(0)
funda['xsga0'] = funda['xsga'].fillna(0)

In [ ]:
# Create simple-just annual Compustat variables

funda['bm'] = funda['ceq']/funda['mve_f']
funda['ep'] = funda['ib']/funda['mve_f']
funda['cashpr'] = ((funda['mve_f']+funda['dltt']-funda['at'])/funda['che'])
funda['dy'] = funda['dvt']/funda['mve_f']
funda['lev'] = funda['lt']/funda['mve_f']
funda['sp'] = funda['sale']/funda['mve_f']
funda['roic'] = (funda['ebit']-funda['nopi'])/(funda['ceq']+funda['lt']-funda['che'])
funda['rd_sale'] = funda['xrd']/funda['sale']
funda['rd_mve'] = funda['xrd']/funda['mve_f']
funda['agr'] = funda['at']/funda.groupby('gvkey')['at'].shift(1) - 1
funda['gma'] = (funda['revt']-funda['cogs'])/funda.groupby('gvkey')['at'].shift(1)
funda['chcsho'] = funda['csho']/funda.groupby('gvkey')['csho'].shift(1) - 1 
funda['lgr'] = funda['lt']/funda.groupby('gvkey')['lt'].shift(1)-1
funda['acc'] = np.where(funda['oancf'].notna(), 
                        (funda['ib']-funda['oancf'])/(funda['at']+funda.groupby('gvkey')['at'].shift(1))/2, 
                        (funda['act']-funda.groupby('gvkey')['act'].shift(1) - (funda['che']-funda.groupby('gvkey')['che'].shift(1))) - ((funda['lct']-funda.groupby('gvkey')['lct'])-(funda['dlc']-funda.groupby('gvkey')['dlc'].shift(1))-(funda['txt']-funda.groupby('gvkey')['txt'].shift(1)-funda['dp']))/((funda['at']+funda.groupby('gvkey')['at'])/2))
funda['pctacc'] = 


